In [5]:
# سلول ۱: وارد کردن کتابخانه‌ها و تنظیمات اولیه مسیرها
import os
import urllib.request
import zipfile
import numpy as np
from scipy.io import wavfile

# تعریف مسیرهای پایه به صورت ثابت (Constant)
DATA_DIR = './data'
ZIP_PATH = os.path.join(DATA_DIR, 'mini_speech_commands.zip')
EXTRACTED_DIR = os.path.join(DATA_DIR, 'mini_speech_commands')

# ساخت پوشه اصلی در صورت عدم وجود
if not os.path.exists(DATA_DIR):
    os.makedirs(DATA_DIR)
    print(f"📁 پوشه پایه ساخته شد: {DATA_DIR}")
else:
    print(f"✅ پوشه پایه آماده است: {DATA_DIR}")

✅ پوشه پایه آماده است: ./data


In [6]:
# سلول ۲: دانلود ایمن و استخراج داده‌ها (با مدیریت خطای فایل خراب)

def is_zip_valid(filepath):
    """بررسی سلامت و یکپارچگی فایل زیپ"""
    try:
        with zipfile.ZipFile(filepath, 'r') as zip_ref:
            return zip_ref.testzip() is None
    except:
        return False

# بررسی اینکه آیا داده‌ها قبلا استخراج شده‌اند یا خیر
if os.path.exists(EXTRACTED_DIR) and len(os.listdir(EXTRACTED_DIR)) > 0:
    print("✅ داده‌ها قبلاً با موفقیت استخراج شده‌اند. نیازی به عملیات مجدد نیست.")
else:
    url = 'http://storage.googleapis.com/download.tensorflow.org/data/mini_speech_commands.zip'
    
    # بررسی و حذف فایل زیپ خراب (اگر وجود داشته باشد)
    if os.path.exists(ZIP_PATH):
        print("🔍 در حال بررسی سلامت فایل زیپ موجود در سیستم...")
        if not is_zip_valid(ZIP_PATH):
            print("⚠️ فایل زیپ خراب است! سیستم در حال حذف و دانلود مجدد است...")
            os.remove(ZIP_PATH)
        else:
            print("✅ فایل زیپ سالم است.")

    # دانلود فایل در صورت عدم وجود
    if not os.path.exists(ZIP_PATH):
        print("⏳ در حال دانلود فایل‌های صوتی (حدود ۱۶۰ مگابایت)...")
        urllib.request.urlretrieve(url, ZIP_PATH)
        print("✅ دانلود با موفقیت انجام شد!")

    # استخراج فایل با بلوک مدیریت خطا
    print("⏳ در حال استخراج فایل‌ها...")
    try:
        with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
            zip_ref.extractall(DATA_DIR)
        print("🎉 استخراج با موفقیت تمام شد! پوشه داده‌ها ایجاد گردید.")
    except zipfile.BadZipFile:
        print("❌ خطا بحرانی: فایل زیپ حتی پس از دانلود مجدد خراب است.")

🔍 در حال بررسی سلامت فایل زیپ موجود در سیستم...
⚠️ فایل زیپ خراب است! سیستم در حال حذف و دانلود مجدد است...
⏳ در حال دانلود فایل‌های صوتی (حدود ۱۶۰ مگابایت)...
✅ دانلود با موفقیت انجام شد!
⏳ در حال استخراج فایل‌ها...
🎉 استخراج با موفقیت تمام شد! پوشه داده‌ها ایجاد گردید.


In [7]:
# سلول ۳: ممیزی داده‌های صوتی (بررسی کیفیت، طول و فایل‌های خراب)

if not os.path.exists(EXTRACTED_DIR):
    print("❌ خطا: پوشه داده‌ها یافت نشد! لطفاً مطمئن شوید سلول ۲ با موفقیت تمام شده است.")
else:
    # استخراج کلمات (فقط پوشه‌ها را در نظر می‌گیریم تا فایل‌های سیستمی مزاحم نشوند)
    words = [d for d in os.listdir(EXTRACTED_DIR) if os.path.isdir(os.path.join(EXTRACTED_DIR, d))]
    print(f"کلمات موجود در مجموعه داده: {words}\n")
    
    if len(words) == 0:
        print("❌ خطا: هیچ پوشه کلمه‌ای در داده‌ها یافت نشد!")
    else:
        first_word = words[0]
        word_dir = os.path.join(EXTRACTED_DIR, first_word)
        print(f"🔍 در حال ممیزی پوشه کلمه نمونه: '{first_word}' ...")

        total_files = 0
        corrupt_files = 0
        durations = []
        sample_rates = set()

        for filename in os.listdir(word_dir):
            if filename.endswith('.wav'):
                total_files += 1
                filepath = os.path.join(word_dir, filename)
                try:
                    # خواندن فایل صوتی
                    sr, data = wavfile.read(filepath)
                    sample_rates.add(sr)
                    
                    # جلوگیری از خطای تقسیم بر صفر
                    if sr > 0:
                        duration = len(data) / sr
                        durations.append(duration)
                except Exception:
                    corrupt_files += 1

        # چاپ گزارش نهایی
        print(f"\n✅ گزارش نهایی ممیزی کلمه '{first_word}':")
        print(f"▫️ تعداد کل فایل‌های بررسی شده: {total_files}")
        if total_files > 0:
            print(f"▫️ نرخ نمونه‌برداری (Sampling Rate): {list(sample_rates)} هرتز")
            print(f"▫️ میانگین طول فایل‌ها: {np.mean(durations):.2f} ثانیه")
            print(f"▫️ کوتاه‌ترین فایل: {np.min(durations):.2f} ثانیه")
            print(f"▫️ طولانی‌ترین فایل: {np.max(durations):.2f} ثانیه")
        print(f"▫️ فایل‌های خراب یا ناخوانا: {corrupt_files}")

کلمات موجود در مجموعه داده: ['down', 'go', 'left', 'no', 'right', 'stop', 'up', 'yes']

🔍 در حال ممیزی پوشه کلمه نمونه: 'down' ...

✅ گزارش نهایی ممیزی کلمه 'down':
▫️ تعداد کل فایل‌های بررسی شده: 1000
▫️ نرخ نمونه‌برداری (Sampling Rate): [16000] هرتز
▫️ میانگین طول فایل‌ها: 0.99 ثانیه
▫️ کوتاه‌ترین فایل: 0.47 ثانیه
▫️ طولانی‌ترین فایل: 1.00 ثانیه
▫️ فایل‌های خراب یا ناخوانا: 0
